# JPMC Consumer Banking - Time Travel & Zero Copy Clone

This notebook demonstrates two powerful Snowflake features:

1. **Time Travel** — Query historical data as it existed at any point within the retention period
2. **Zero Copy Clone** — Instantly create full copies of tables, schemas, or databases without duplicating storage

**Database:** `JPMC_CONSUMER_BANKING_HOL.HOL_LAB`  
**Warehouse:** `HOL_MEDIUM_WH`  
**Tables:** ACCOUNTS, LOANS, CREDIT_CARDS

In [ ]:
-- =============================================================================
-- SETUP
-- =============================================================================

USE ROLE ACCOUNTADMIN;
USE WAREHOUSE HOL_MEDIUM_WH;
USE SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB;

## Part 1: Time Travel

Snowflake automatically retains historical versions of data for a configurable retention period (up to 90 days on Enterprise+). This enables:
- Querying data as of a past timestamp (`AT`)
- Querying data before a specific change (`BEFORE`)
- Restoring accidentally deleted or modified data
- Auditing changes over time

In [ ]:
-- =============================================================================
-- Step 1: Check current state and retention settings
-- =============================================================================

-- View current row count and data retention setting
SELECT COUNT(*) AS current_row_count FROM ACCOUNTS;

SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN TABLE ACCOUNTS;

In [ ]:
-- =============================================================================
-- Step 2: Make a change — simulate an accidental UPDATE
-- =============================================================================

-- Record the current timestamp before making changes
SET ts_before_update = CURRENT_TIMESTAMP();

-- Check current balances for a few accounts
SELECT ACCOUNT_ID, CUSTOMER_ID, BALANCE, STATUS
FROM ACCOUNTS
WHERE STATUS = 'ACTIVE'
LIMIT 5;

-- Simulate accidental update: set all active accounts to zero balance
UPDATE ACCOUNTS
SET BALANCE = 0
WHERE STATUS = 'ACTIVE';

SELECT 'Rows updated' AS action, COUNT(*) AS affected_rows
FROM ACCOUNTS WHERE STATUS = 'ACTIVE' AND BALANCE = 0;

In [ ]:
-- =============================================================================
-- Step 3: Use Time Travel to see data BEFORE the accident
-- =============================================================================

-- Query the table as it was before the update
SELECT ACCOUNT_ID, CUSTOMER_ID, BALANCE, STATUS
FROM ACCOUNTS AT(TIMESTAMP => $ts_before_update)
WHERE STATUS = 'ACTIVE'
LIMIT 5;

-- Compare current vs historical: how much balance was lost?
SELECT
    'Before Update' AS state,
    SUM(BALANCE) AS total_balance,
    COUNT(*) AS active_accounts
FROM ACCOUNTS AT(TIMESTAMP => $ts_before_update)
WHERE STATUS = 'ACTIVE'
UNION ALL
SELECT
    'After Update' AS state,
    SUM(BALANCE) AS total_balance,
    COUNT(*) AS active_accounts
FROM ACCOUNTS
WHERE STATUS = 'ACTIVE';

In [ ]:
-- =============================================================================
-- Step 4: Restore the data using Time Travel
-- =============================================================================

-- Restore using CREATE OR REPLACE from time travel
CREATE OR REPLACE TABLE ACCOUNTS AS
SELECT * FROM ACCOUNTS AT(TIMESTAMP => $ts_before_update);

-- Verify restoration
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN BALANCE = 0 AND STATUS = 'ACTIVE' THEN 1 ELSE 0 END) AS zero_balance_active,
    SUM(BALANCE) AS total_balance
FROM ACCOUNTS;

In [ ]:
-- =============================================================================
-- Step 5: Time Travel with OFFSET (relative time)
-- =============================================================================

-- Query data as it was 5 minutes ago (offset in seconds)
SELECT COUNT(*) AS row_count_5min_ago
FROM ACCOUNTS AT(OFFSET => -60*5);

-- Query using a specific statement ID (useful for undoing a known query)
-- BEFORE keyword gives data just before that statement executed
-- SELECT * FROM ACCOUNTS BEFORE(STATEMENT => '<query_id>');

In [ ]:
-- =============================================================================
-- Step 6: Time Travel on DROPPED tables (UNDROP)
-- =============================================================================

-- Drop a table
DROP TABLE ACCOUNTS;

-- Oops! Restore it with UNDROP
UNDROP TABLE ACCOUNTS;

-- Verify it's back
SELECT COUNT(*) AS restored_rows FROM ACCOUNTS;

## Part 2: Zero Copy Clone

Zero Copy Clone creates an instant, metadata-only copy of a table, schema, or database. Key properties:
- **No additional storage cost** at creation time (shares underlying micro-partitions)
- **Independent after creation** — changes to clone don't affect source and vice versa
- **Storage only grows** when cloned data diverges (copy-on-write)
- **Supports** tables, schemas, databases, stages, file formats, sequences, streams, and tasks

In [ ]:
-- =============================================================================
-- Step 7: Clone a table (instant, zero storage)
-- =============================================================================

-- Clone the ACCOUNTS table
CREATE OR REPLACE TABLE ACCOUNTS_CLONE CLONE ACCOUNTS;

-- Verify: same data, zero additional storage
SELECT
    'ACCOUNTS' AS table_name, COUNT(*) AS rows FROM ACCOUNTS
UNION ALL
SELECT
    'ACCOUNTS_CLONE', COUNT(*) FROM ACCOUNTS_CLONE;

-- Check storage — clone initially uses 0 bytes
SELECT TABLE_NAME, ACTIVE_BYTES, TIME_TRAVEL_BYTES, RETAINED_FOR_CLONE_BYTES
FROM INFORMATION_SCHEMA.TABLE_STORAGE_METRICS
WHERE TABLE_NAME IN ('ACCOUNTS', 'ACCOUNTS_CLONE')
AND TABLE_CATALOG = 'JPMC_CONSUMER_BANKING_HOL'
AND TABLE_SCHEMA = 'HOL_LAB';

In [ ]:
-- =============================================================================
-- Step 8: Modify the clone (independent from source)
-- =============================================================================

-- Delete some rows from the clone — source table is NOT affected
DELETE FROM ACCOUNTS_CLONE WHERE STATUS != 'ACTIVE';

-- Add a derived column to the clone
ALTER TABLE ACCOUNTS_CLONE ADD COLUMN BALANCE_TIER VARCHAR;

UPDATE ACCOUNTS_CLONE
SET BALANCE_TIER = CASE
    WHEN BALANCE >= 100000 THEN 'PREMIUM'
    WHEN BALANCE >= 25000 THEN 'GOLD'
    WHEN BALANCE >= 5000 THEN 'SILVER'
    ELSE 'BASIC'
END;

-- Compare: source unchanged, clone modified
SELECT 'SOURCE' AS table_type, COUNT(*) AS rows, NULL AS has_balance_tier
FROM ACCOUNTS
UNION ALL
SELECT 'CLONE', COUNT(*), 'YES'
FROM ACCOUNTS_CLONE;

In [ ]:
-- =============================================================================
-- Step 9: Clone an entire SCHEMA (all tables at once)
-- =============================================================================

-- Clone the full schema for dev/testing — instant, no storage duplication
CREATE OR REPLACE SCHEMA HOL_LAB_DEV CLONE HOL_LAB;

-- Verify all tables were cloned
SHOW TABLES IN SCHEMA JPMC_CONSUMER_BANKING_HOL.HOL_LAB_DEV;

-- Query from the cloned schema
SELECT COUNT(*) AS dev_accounts FROM JPMC_CONSUMER_BANKING_HOL.HOL_LAB_DEV.ACCOUNTS;

In [ ]:
-- =============================================================================
-- Step 10: Clone with Time Travel (point-in-time clone)
-- =============================================================================

-- Clone the ACCOUNTS table as it existed 5 minutes ago
CREATE OR REPLACE TABLE ACCOUNTS_HISTORICAL_CLONE
    CLONE ACCOUNTS AT(OFFSET => -60*5);

SELECT COUNT(*) AS historical_clone_rows FROM ACCOUNTS_HISTORICAL_CLONE;

In [ ]:
-- =============================================================================
-- Step 11: Use Cases Summary
-- =============================================================================

-- Real-world use cases demonstrated:
--
-- TIME TRAVEL:
--   1. Undo accidental DML (UPDATE/DELETE) without backup restores
--   2. Audit data changes over time
--   3. Recover dropped tables (UNDROP)
--   4. Point-in-time reporting
--
-- ZERO COPY CLONE:
--   1. Instant dev/test environments from production data
--   2. Safe experimentation without affecting source
--   3. Point-in-time snapshots (clone + time travel)
--   4. Data sandboxes for analytics teams
--   5. Pre-release testing of schema changes

SELECT 'Time Travel + Zero Copy Clone Demo Complete' AS status;

In [ ]:
-- =============================================================================
-- CLEANUP (optional)
-- =============================================================================

DROP TABLE IF EXISTS ACCOUNTS_CLONE;
DROP TABLE IF EXISTS ACCOUNTS_HISTORICAL_CLONE;
DROP SCHEMA IF EXISTS JPMC_CONSUMER_BANKING_HOL.HOL_LAB_DEV;

SELECT 'Cleanup complete' AS status;